# Fine-tune AraBERT to detect literal Arabic translations

Runs in Google Colab: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

It uses the same data and the same grouped folds as `baseline.py`, so the numbers are comparable.

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" scikit-learn pandas
!git clone -q https://github.com/ShahadA-lmutairi/LinguAlyzer.git
%cd LinguAlyzer/naturalness

In [ ]:
import numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupKFold
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)
from common import load_pairs, to_rows

MODEL = "aubmindlab/bert-base-arabertv02"   # also try: "CAMeL-Lab/bert-base-arabic-camelbert-msa"
set_seed(42)
rows = to_rows(load_pairs())
print(len(rows), "sentences,", rows["group"].nunique(), "pairs")
rows.head()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def encode(df):
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=64), batched=True)

def run_fold(train_df, test_df):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    args = TrainingArguments(output_dir="out", num_train_epochs=6, learning_rate=3e-5,
                             per_device_train_batch_size=16, weight_decay=0.01,
                             save_strategy="no", logging_steps=10, report_to=[], seed=42)
    trainer = Trainer(model=model, args=args, train_dataset=encode(train_df), tokenizer=tokenizer)
    trainer.train()
    logits = trainer.predict(encode(test_df)).predictions
    return logits.argmax(-1)

In [ ]:
X, y, groups = rows["text"].values, rows["label"].values, rows["group"].values
k = min(5, len(set(groups)))
acc, f1, wrong = [], [], []
for fold, (tr, te) in enumerate(GroupKFold(n_splits=k).split(X, y, groups), 1):
    pred = run_fold(rows.iloc[tr], rows.iloc[te])
    acc.append(accuracy_score(y[te], pred)); f1.append(f1_score(y[te], pred, average="macro"))
    test = rows.iloc[te].assign(pred=pred)
    wrong.append(test[test.pred != test.label])
    print(f"fold {fold}: accuracy {acc[-1]:.1%}, macro-F1 {f1[-1]:.1%}")

print(f"\nAraBERT: accuracy {np.mean(acc):.1%} ± {np.std(acc):.1%}, macro-F1 {np.mean(f1):.1%} ± {np.std(f1):.1%}")

## Error analysis

Which sentences did the model get wrong? This is often more useful than the score: it shows which issue types need more examples.

In [ ]:
errors = pd.concat(wrong)
errors["label"] = errors["label"].map({1: "literal", 0: "natural"})
errors["pred"] = errors["pred"].map({1: "literal", 0: "natural"})
errors[["text", "label", "pred", "issue"]]

## Record the result

Copy the AraBERT line into `results/baseline.md` (or a new `results/arabert.md`) together with the date, the model name and the number of pairs.